In [1]:
import os
#os.environ["HF_HOME"] = "/projectnb/vkolagrp/skowshik/.cache/"


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

device: cuda


In [4]:
!echo $HF_HOME

/projectnb/cs599m1/students/micahb/huggingface


In [5]:
model_id = "Qwen/Qwen2.5-3B-Instruct"
n_devices = 1

In [6]:
# load model
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    #cache_dir = "/projectnb/vkolagrp/skowshik/.cache/",
    torch_dtype="auto",
    device_map="auto")

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:

# define a hook function that caches activations
def cache_hook(cache):
	def hook(module, input, output):
		cache.append(output[0]) # the output of the residual stream is actually a tuple, where the first entry is the activation
	return hook



In [ ]:
def make_cache_hook(layer_idx, cache):
    def hook(module, inputs, output):
        act = output[0] if isinstance(output, tuple) else output
        cache[layer_idx] = act.detach().cpu()
    return hook

In [8]:
model.model.layers

ModuleList(
  (0-35): 36 x Qwen2DecoderLayer(
    (self_attn): Qwen2Attention(
      (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
      (k_proj): Linear(in_features=2048, out_features=256, bias=True)
      (v_proj): Linear(in_features=2048, out_features=256, bias=True)
      (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
    )
    (mlp): Qwen2MLP(
      (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
      (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
      (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
    (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
  )
)

In [15]:
# Loop over all layers and attention heads in the model
num_layers = len(model.transformer.h)
# Try to infer the number of heads by inspecting a typical transformer block
example_block = model.transformer.h[0]
if hasattr(example_block.self_attn, 'num_heads'):
    num_heads = example_block.self_attn.num_heads
elif hasattr(example_block.self_attn, 'num_attention_heads'):
    num_heads = example_block.self_attn.num_attention_heads
else:
    # As a fallback, inspect the attention projections
    # (e.g., use q_proj weight shape [embed_dim, hidden_dim])
    # For Qwen-style, let's try:
    num_heads = model.config.num_attention_heads if hasattr(model.config, "num_attention_heads") else None

print(f"Total layers: {num_layers}")
print(f"Heads per layer: {num_heads}")

for layer_idx in range(num_layers):
    block = model.transformer.h[layer_idx]
    print(f"\nLayer {layer_idx}:")
    for head_idx in range(num_heads):
        print(f"  Head {head_idx}")
        # At this point, you can hook attention head activations, query params, etc., 
        # e.g. block.self_attn; details depend on where/what you wish to analyze


AttributeError: 'Qwen2ForCausalLM' object has no attribute 'transformer'

In [9]:
import pandas as pd

df = pd.read_json("../data/generated_examples/occupations.jsonl", lines=True)

i_prompts = df["i"]
you_prompts = df["you"]

print(i_prompts[0])

I am a firefighter. Explain how to use a fire extinguisher.


In [36]:
import torch, gc

gc.collect()                 # run Python’s garbage collector
torch.cuda.empty_cache()     # release unreferenced cached memory
torch.cuda.ipc_collect()     # (optional) reclaim interprocess memory
del cache

In [10]:
# define layer to do the activation steering on - Qwen2.5 Layer 14 to choose an arbitrary middle layer
layer_id = 14

i_prompts = i_prompts[:100]
# get internal activations at layer 14 over all of our prompts
cache = []
for prompt in i_prompts: 
    handle = model.model.layers[layer_id].register_forward_hook(cache_hook(cache))

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    _ = model(**inputs)

    handle.remove()  # it's very important to keep track of hook handles and remove the hooks 



In [26]:
import numpy as np

activations_i = [act[-1, :] for act in cache]
mean_i = np.mean(activations_i)

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [19]:
import pandas as pd

df = pd.read_json("../data/generated_examples/occupations.jsonl", lines=True)

i_prompts = df["i"]
you_prompts = df["you"]

print(i_prompts[0])

I am a firefighter. Explain how to use a fire extinguisher.


In [ ]:


model.generate()

AttributeError: 'str' object has no attribute 'shape'